# Coherent MXene/metal interface builder

This page runs the **coherentmxmetal** Sim2L: it builds a metal (111) slab on an MXene (0001) in one orthogonal
box, with the smallest coherent supercell within the strain tolerance, and can relax it with a MACE foundation model.
Runs with the same inputs come from the nanoHUB cache.

Run the cells in order. Change the inputs in step 2 to build other interfaces.
Source and documentation: https://github.com/docampomill/coherent-interface-builder

## 1. Find the Sim2L and list its inputs

In [ ]:
from simtool import searchForSimTool, getSimToolInputs, getSimToolOutputs, Run

loc = searchForSimTool("coherentmxmetal")
inputs = getSimToolInputs(loc)
inputs

## 2. Set the inputs

Lengths take units, e.g. `"0.25 nm"`. Inputs not set here keep their defaults.

In [ ]:
inputs.mxene.value = "V2C"
inputs.stacking.value = "ABC"
inputs.metal.value = "Au"
inputs.metal_layers.value = 4
inputs.gap.value = "2.3 angstrom"
inputs.run_relaxation.value = False      # True: relax with MACE (slower; set max_steps, fmax, ...)

## 3. Run

In [ ]:
r = Run(loc, inputs)

## 4. Results

In [ ]:
match = r.read("match")
print(r.read("formula"), f"{r.read('n_atoms')} atoms, box", r.read("box"), "Å")
for axis in "xy":
    m = match[axis]
    print(f"{axis}: {m['mxene']} MXene : {m['metal']} metal cells, {m['length']} Å, metal strain {m['strain']:+.2%}")
if inputs.run_relaxation.value:
    print("energy", r.read("energy"), "eV, relaxed gap", r.read("gap_relaxed"), "Å, work of adhesion",
          r.read("work_of_adhesion"), "J/m²")

In [ ]:
# Side and top views of the interface (relaxed, if it was relaxed)
import io
import matplotlib.pyplot as plt
from ase.io import read
from ase.visualize.plot import plot_atoms

atoms = read(io.StringIO(r.read("POSCAR_relaxed") or r.read("POSCAR")), format="vasp")
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
plot_atoms(atoms.repeat((3, 3, 1)), ax[0], rotation="-90x", radii=0.5)
plot_atoms(atoms.repeat((3, 3, 1)), ax[1], radii=0.5)
ax[0].set_title("side (xz)")
ax[1].set_title("top (xy)")
for a in ax:
    a.set_axis_off()

## 5. Save the structures

The POSCARs are written next to this notebook; download them from the file browser.

In [ ]:
from pathlib import Path

for name in ["POSCAR", "POSCAR_mxene", "POSCAR_metal", "POSCAR_relaxed"]:
    text = r.read(name)
    if text:
        Path(name).write_text(text)
        print("wrote", name)